# Separator runtime benchmark grid

This notebook loads benchmark CSV outputs, summarizes runtimes over a grid of $(k,m)$ values, and plots direct runtime comparisons between methods.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

RESULT_DIR = 'benchmark_results'
SUMMARY_CSV = os.path.join(RESULT_DIR, 'benchmark_summary.csv')
DETAILED_CSV = os.path.join(RESULT_DIR, 'benchmark_detailed.csv')

summary = pd.read_csv(SUMMARY_CSV)
detailed = pd.read_csv(DETAILED_CSV)

print('Loaded summary rows:', len(summary))
print('Loaded detailed rows:', len(detailed))
summary.head()

## Summary table

In [ ]:
summary_sorted = summary.sort_values(['method', 'm', 'k']).reset_index(drop=True)
summary_sorted

## Runtime heatmaps by method

In [ ]:
methods = list(summary['method'].unique())
fig, axes = plt.subplots(1, len(methods), figsize=(6 * max(1, len(methods)), 5), squeeze=False)

for ax, method in zip(axes[0], methods):
    subset = summary[summary['method'] == method]
    pivot = subset.pivot(index='m', columns='k', values='mean_runtime_s').sort_index().sort_index(axis=1)
    im = ax.imshow(pivot.values, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'{method}: mean runtime (s)')
    ax.set_xlabel('k')
    ax.set_ylabel('m')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f'{pivot.values[i, j]:.3f}', ha='center', va='center', color='white', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Direct runtime ratio between two methods

In [ ]:
if len(methods) >= 2:
    ref_method = methods[0]
    cmp_method = methods[1]

    ref = summary[summary['method'] == ref_method][['m', 'k', 'mean_runtime_s']].rename(columns={'mean_runtime_s': 'ref_runtime_s'})
    cmp = summary[summary['method'] == cmp_method][['m', 'k', 'mean_runtime_s']].rename(columns={'mean_runtime_s': 'cmp_runtime_s'})
    merged = ref.merge(cmp, on=['m', 'k'], how='inner')
    merged['runtime_ratio'] = merged['cmp_runtime_s'] / merged['ref_runtime_s']

    ratio_pivot = merged.pivot(index='m', columns='k', values='runtime_ratio').sort_index().sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(ratio_pivot.values, aspect='auto', origin='lower', cmap='coolwarm')
    ax.set_title(f'Runtime ratio = {cmp_method} / {ref_method}')
    ax.set_xlabel('k')
    ax.set_ylabel('m')
    ax.set_xticks(range(len(ratio_pivot.columns)))
    ax.set_xticklabels(ratio_pivot.columns)
    ax.set_yticks(range(len(ratio_pivot.index)))
    ax.set_yticklabels(ratio_pivot.index)
    for i in range(len(ratio_pivot.index)):
        for j in range(len(ratio_pivot.columns)):
            ax.text(j, i, f'{ratio_pivot.values[i, j]:.2f}x', ha='center', va='center', color='black', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    merged.sort_values(['m', 'k']).reset_index(drop=True)
else:
    print('Add a second method to see direct runtime ratios.')

## Detailed runtime distributions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

positions = np.arange(len(methods))
all_values = [detailed[detailed['method'] == method]['runtime_s'].values for method in methods]
ax.boxplot(all_values, tick_labels=methods, showfliers=False)
ax.set_title('Runtime distribution by method')
ax.set_ylabel('runtime (s)')
plt.tight_layout()
plt.show()